# Demo ACL con Foundry IQ (permessi a livello di documento)

Il notebook mostra che una knowledge base Foundry IQ restituisce a ogni utente **solo i documenti che puo' leggere nella sorgente**. Le ACL POSIX di ADLS Gen2 vengono importate in fase di ingestion (`ingestionPermissionOptions: ["userIds", "groupIds"]`) e applicate in fase di query con il token dell'utente nell'header `x-ms-query-source-authorization`.

| Cartella (container `acl-docs`) | Documento | Gruppi con accesso |
|---|---|---|
| `pubblico/` | Procedura richiesta ricambi (codice **PRC-100**) | Tecnici, Direzione |
| `manutenzione-riservata/` | Codici modalita' service (**SRV-7741-ALFA**) | Tecnici |
| `direzione-riservata/` | Accordi quadro fornitori 2027 (sconto **18%**) | Direzione |

Dopo il setup l'utente corrente e' membro di **Tecnici** e owner di entrambi i gruppi, quindi puo' cambiare la propria appartenenza dal notebook.

## Prerequisiti

1. `azd up` completato: il postprovision esegue [scripts/setup_acl_demo.ps1](scripts/setup_acl_demo.ps1) (gruppi Entra, upload e ACL) e il postdeploy crea `acl-docs-ks` e `wpc-acl-kb` con [scripts/setup_foundry_iq.ps1](scripts/setup_foundry_iq.ps1).
2. Kernel sul `.venv` della root del repo (creato dal postprovision) e `az login` sullo stesso tenant delle risorse.
3. Dopo la creazione della knowledge source attendi qualche minuto: l'indexer generato deve completare la prima ingestion.

## 1. Configurazione

I valori vengono letti da `azd env get-values`. Le domande sono pensate per colpire un solo documento ciascuna.

In [ ]:
import json
import shutil
import subprocess
import time

import pandas as pd
import requests
from azure.identity import AzureCliCredential, DeviceCodeCredential

AZD_ENVIRONMENT = None  # None = ambiente azd predefinito
API_VERSION = "2026-11-01-preview"

azd_args = [shutil.which("azd") or "azd", "env", "get-values", "--output", "json"]
if AZD_ENVIRONMENT:
    azd_args += ["--environment", AZD_ENVIRONMENT]
env = json.loads(subprocess.run(azd_args, capture_output=True, text=True, check=True).stdout)

SEARCH_ENDPOINT = env["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT"].rstrip("/")
TENANT_ID = env["AZURE_TENANT_ID"]
KB_NAME = env.get("FOUNDRYIQ_ACL_KB_NAME", "wpc-acl-kb")
KS_NAME = "acl-docs-ks"
DFS_ENDPOINT = env["FOUNDRYIQ_ACL_DFS_ENDPOINT"]
CONTAINER = env.get("FOUNDRYIQ_ACL_CONTAINER", "acl-docs")
GROUPS = {
    "Tecnici": env["FOUNDRYIQ_ACL_GROUP_TECNICI_ID"],
    "Direzione": env["FOUNDRYIQ_ACL_GROUP_DIREZIONE_ID"],
}

QUESTIONS = {
    "pubblico": "Qual e' il codice della procedura standard di richiesta ricambi?",
    "manutenzione-riservata": "Qual e' il codice della modalita service del Ven Systems Model 12?",
    "direzione-riservata": "Quale sconto quadro abbiamo con Delta Systems nel 2027?",
}
print(f"Search: {SEARCH_ENDPOINT}\nKnowledge base: {KB_NAME}\nGruppi: {GROUPS}")

## 2. Identita' e helper di retrieve

Ogni chiamata usa due token distinti:

- `Authorization`: chi **chiama** Search (qui l'utente demo, con il ruolo *Search Index Data Contributor*). Puo' essere anche un'app o una managed identity.
- `x-ms-query-source-authorization`: l'utente **di cui valutare i permessi** sui documenti. Non richiede ruoli su Search.

Senza il secondo header restano visibili solo i documenti con grant pubblici: qui nessuno, perche' tutte le cartelle hanno ACL nominative.

In [ ]:
SEARCH_SCOPE = "https://search.azure.com/.default"
credential = AzureCliCredential(tenant_id=TENANT_ID)
me = json.loads(subprocess.run([shutil.which("az") or "az", "ad", "signed-in-user", "show", "--query", "{id:id, upn:userPrincipalName}", "-o", "json"],
                               capture_output=True, text=True, check=True).stdout)
print(f"Utente corrente: {me['upn']} ({me['id']})")


def retrieve(question: str, user_token: str | None) -> dict:
    headers = {
        "Authorization": f"Bearer {credential.get_token(SEARCH_SCOPE).token}",
        "Content-Type": "application/json",
    }
    if user_token:
        headers["x-ms-query-source-authorization"] = user_token
    body = {"intents": [{"type": "semantic", "search": question}], "includeActivity": True}
    url = f"{SEARCH_ENDPOINT}/knowledgebases/{KB_NAME}/retrieve?api-version={API_VERSION}"
    response = requests.post(url, headers=headers, json=body, timeout=120)
    if not response.ok:
        raise RuntimeError(f"retrieve {response.status_code}: {response.text}")
    return response.json()


def documents(result: dict) -> list[str]:
    names = []
    for ref in result.get("references") or []:
        url = ref.get("blobUrl") or ref.get("docKey") or ref.get("id") or ""
        names.append(url.split(f"/{CONTAINER}/", 1)[-1])
    return sorted(set(names))


def run_matrix(label: str, user_token: str | None) -> pd.DataFrame:
    rows = []
    for folder, question in QUESTIONS.items():
        docs = documents(retrieve(question, user_token))
        rows.append({
            "identita'": label,
            "domanda su": folder,
            "documento atteso trovato": any(d.startswith(folder + "/") for d in docs),
            "documenti restituiti": ", ".join(docs) or "(nessuno)",
        })
    return pd.DataFrame(rows)


def user_token() -> str:
    return credential.get_token(SEARCH_SCOPE).token

## 3. Stato della sorgente e ACL di origine

La knowledge source genera indexer, data source e indice. Verifica che l'ingestion sia completata, poi leggi le ACL effettive di ogni cartella su ADLS Gen2: sono la "verita'" che Foundry IQ replica nell'indice.

In [ ]:
status = requests.get(
    f"{SEARCH_ENDPOINT}/knowledgesources/{KS_NAME}/status?api-version={API_VERSION}",
    headers={"Authorization": f"Bearer {credential.get_token(SEARCH_SCOPE).token}"},
    timeout=60,
)
print(json.dumps(status.json(), indent=2) if status.ok else f"Stato non disponibile ({status.status_code}): {status.text}")

In [ ]:
from azure.storage.filedatalake import DataLakeServiceClient

names_by_id = {v: k for k, v in GROUPS.items()}
fs = DataLakeServiceClient(DFS_ENDPOINT, credential=credential).get_file_system_client(CONTAINER)
rows = []
for folder in QUESTIONS:
    acl = fs.get_directory_client(folder).get_access_control()["acl"]
    groups = [names_by_id.get(e.split(":")[1], e.split(":")[1]) for e in acl.split(",")
              if e.startswith("group:") and e.split(":")[1] and not e.startswith("default:")]
    rows.append({"cartella": folder, "gruppi con accesso": ", ".join(groups)})
pd.DataFrame(rows)

## 4. Senza identita' utente

Solo `Authorization`, nessun `x-ms-query-source-authorization`: il motore applica comunque il filtro e non restituisce documenti protetti. Il ruolo RBAC del chiamante su Search **non** concede accesso ai contenuti.

In [ ]:
run_matrix("nessun utente", None)

## 5. Utente corrente (membro di Tecnici)

Atteso: trova la procedura pubblica e i codici service, **non** gli accordi della direzione.

In [ ]:
baseline = run_matrix(f"{me['upn']} (Tecnici)", user_token())
baseline

## 6. Cambio di appartenenza ai gruppi, senza reindicizzare

I gruppi vengono risolti a **query time** tramite Microsoft Graph, quindi aggiungere l'utente a *Direzione* basta a rendere visibile il documento riservato: non serve modificare ACL o rieseguire l'indexer. La propagazione dell'appartenenza e la cache possono richiedere alcuni minuti; se il risultato non cambia, riesegui la cella.

In [ ]:
AZ = shutil.which("az") or "az"
subprocess.run([AZ, "ad", "group", "member", "add", "--group", GROUPS["Direzione"], "--member-id", me["id"]], check=False)
print("Aggiunto a Direzione, attendo 60 s per la propagazione...")
time.sleep(60)
run_matrix(f"{me['upn']} (Tecnici + Direzione)", user_token())

In [ ]:
# Ripristina lo stato iniziale (solo Tecnici).
subprocess.run([AZ, "ad", "group", "member", "remove", "--group", GROUPS["Direzione"], "--member-id", me["id"]], check=False)
print("Rimosso da Direzione.")

## 7. (Opzionale) Secondo utente con device code

Accedi con un altro utente dello stesso tenant, ad esempio un membro solo di *Direzione*: il chiamante resta l'utente demo, mentre i permessi valutati sono quelli del secondo utente, che non ha bisogno di ruoli su Azure. Aggiungilo prima al gruppo, per esempio con `az ad group member add --group <id-direzione> --member-id <object-id>`.

In [ ]:
RUN_SECOND_USER = False

if RUN_SECOND_USER:
    second_user = DeviceCodeCredential(tenant_id=TENANT_ID)
    second_token = second_user.get_token(SEARCH_SCOPE).token
    display(run_matrix("secondo utente", second_token))
else:
    print("Imposta RUN_SECOND_USER = True per eseguire il test con un secondo utente.")

## Note

- Il filtro vale per la knowledge base e per l'endpoint MCP della KB solo se il client passa il token utente. Un agente Foundry che usa la connessione con l'identita' gestita del progetto **non** porta l'identita' dell'utente: per l'enforcement end-to-end il chiamante deve inoltrare il token delegato.
- Le modifiche alle ACL di ADLS Gen2 vengono riallineate dall'indexer circa una volta al giorno, oppure con un resync dei permessi. Le modifiche all'appartenenza ai gruppi invece hanno effetto a query time.
- Sono supportate solo voci ACL nominative (utenti e gruppi). Owner, owning group e `other` vengono ignorati.
- Riferimenti: [ACL ADLS Gen2](https://learn.microsoft.com/azure/search/search-indexer-access-control-lists-and-role-based-access), [enforcement in query](https://learn.microsoft.com/azure/search/search-query-access-control-rbac-enforcement), [retrieve con autorizzazione](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve).